# Transverse field Ising model using NetKet

This notebook follows the NetKet tutorial, ``https://netket.readthedocs.io/en/latest/tutorials/gs-ising.html``. It studies the tranvserve field Ising model, the same Hamiltonian

$$
H = V\sum_{i=1}^{N-1} \sigma_i^z \sigma_{i+1}^z
     + \Gamma\sum_{i=1}^{N} \sigma_i^x,
\qquad
J,\;h \in \mathbb{R},
\qquad
J > 0,\; h \geq 0
$$

which was explored using exact diagonalization in this repository under `notebooks/exact_diagonalization.ipynb`.

The boundary conditions are periodic with lattice sites $i = 0, 1, \cdots , L-1$ with $i = L \equiv 0$.

First, we define the Hilbert space $\mathcal{H}$

In [3]:
import netket as nk
import jax
from netket.operator.spin import sigmax, sigmaz

N = 10
hi = nk.hilbert.Spin(s=1 / 2, N=N)

# Initialize random basis vectors
hi.random_state(jax.random.key(0), 3)

# Take \Gamma = -1. Construct the transverse part of H
Gamma = -1
H = sum([Gamma * sigmax(hi, i) for i in range(N)])

# Take V = -1, and construct interaction terms of H
V = -1
H += sum([V * sigmaz(hi, i) @ sigmaz(hi, (i + 1) % N) for i in range(N)])

# Convert to sparse matrix to perform exact diagonalization
sp_h = H.to_sparse()

from scipy.sparse.linalg import eigsh

eig_vals, eig_vecs = eigsh(sp_h, k=2, which="SA")

print("eigenvalues with scipy sparse:", eig_vals)

E_gs = eig_vals[0]

/Users/fionncurran/capstone/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


∣NK⟩ Tip: To make sure your Markov chains are well-mixed, call vstate.thermalise(H) after loading a checkpoint.

eigenvalues with scipy sparse: [-12.78490644 -12.62750303]


Here we copy the bespoke implementation of ED from `exact_diagonlization.ipynb`, modified to have periodic boundary conditions, to compare to the NetKet + scipy sparse diagonalization

In [4]:
import jax.numpy as jnp

# Define Pauli matrices
sigma_x = jnp.array([[0, 1], 
                     [1, 0]])
sigma_z = jnp.array([[1, 0], 
                     [0, -1]])

I = jnp.eye(2)

def act_on_site(operator, site, num_sites):
    result = 1

    for i in range(num_sites):
        if i == site:
            result = jnp.kron(result, operator) 
        else:
            result = jnp.kron(result, I)
    return result

def tf_ising(N, J, h):
    dim = 2 ** N
    H = jnp.zeros((dim, dim)) # Empty 2^n square matrix

    # Spin coupling terms
    """ Old implementation has open boundaries: no coupling of site i = N to i = 1
    for i in range(N-1):
        H -= J * act_on_site(sigma_z, i, N) @ act_on_site(sigma_z, i+1, N)
    """

    for i in range(N):
        H -= J * act_on_site(sigma_z, i, N) @ act_on_site(sigma_z, (i+1) % N, N)

    # Transverse field terms
    for i in range(N):
        H -= h * act_on_site(sigma_x, i, N)

    return H

# Parameters
N = 10
J = 1.0
h = 1.0

H = tf_ising(N, J, h)

E, V = jnp.linalg.eigh(H)

eig_vals = [float(E[0]), float(E[1])]

print("eigenvalues with bespoke ED (first 2):", eig_vals)


eigenvalues with bespoke ED (first 2): [-12.784906442999318, -12.627503029350088]


Results for N=10 sites:

$$
\text{eigenvalues with scipy sparse:}\qquad [-12.78490644 -12.62750303] \\
\text{eigenvalues with bespoke ED (first 2):}\qquad [-12.784906442999318, -12.627503029350088]
$$

Runtime for sparse ED / NetKet was $< 0.0$ s vs. $0.7$ s for original implementation.

## Variational approximation of the ground state

The tutorial introduces a "mean field" ansatz,

$$
\langle \sigma_1^z,\cdots, \sigma_N^z | \Psi_{\text{mf}} \rangle = \Pi_{i=1}^N \Phi(\sigma_i^z),
$$

with the single-spin wavefunctions as the variational "parameters" (i.e. the parameterization of these functions). Normalization is an additional constraint

$$
| \Phi (\uparrow) |^2 + | \Phi (\downarrow) |^2 = 1,
$$

and also assumes $\Phi(\sigma^z) = \sqrt{P(\sigma^z)}$ taking the phase $\phi=0$ justified by the fact that the ground state in this model is real and positive. Then it assumes a further ansatz for $P$,

$$
P(\sigma^z; \lambda) = \frac{1}{1+e^{-\lambda \sigma^z}}
$$

In [7]:
from flax import nnx

class MF(nnx.Module):
    """
    A class implementing a uniform mean-field model.
    """

    def __init__(self, *, rngs: nnx.Rngs):
        key = rngs.params()
        self.log_phi_local = nnx.Param(jax.random.normal(key, (1,)))

    def __call__(self, x: jax.Array):
        p = nnx.log_sigmoid(self.log_phi_local * x)

        return 0.5 * jnp.sum(p, axis=-1)

mf_model = MF(rngs=nnx.Rngs(0))

sampler = nk.sampler.MetropolisLocal(hi)

vstate = nk.vqs.MCState(sampler, mf_model, n_samples=512)

print(vstate.parameters)

{'log_phi_local': Array([1.88002989], dtype=float64)}
